<a href="https://colab.research.google.com/github/Maxim7007/RuMedJ-2016-2026-v4_1/blob/main/RuMedJ_2016_2026_v4_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ============================================================
# ИТОГОВЫЙ СКРИПТ (ВЕРСИЯ 4.0.2)
# Исследование: Профилирование российских медицинских журналов в Scopus (2016–2025)
# Включает: фильтрацию, динамику, тематики, журналы, самоцитирование, верификацию, прирост журналов
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, pearsonr
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# 1. ЗАГРУЗКА И ФИЛЬТРАЦИЯ ДАННЫХ
# ============================================================

print("="*60)
print("1. ЗАГРУЗКА И ФИЛЬТРАЦИЯ ДАННЫХ")
print("="*60)

print("Загрузка основного датасета...")
main_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vSETe-0hT6Y3TEGP5GkDIqhwqQbcyF0IYpRO1D3yainm2Hvc3rX-CiDQLoMPgQSVxmbAnmVheCNH5_q/pub?output=csv'
main_df = pd.read_csv(main_url)
print(f"Основной датасет загружен. Размер: {main_df.shape}")

print("Загрузка экспертного списка...")
expert_url = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRHuSNBoyXrjtzZInQ5UXrFimVJ80m9AOvquKlg8BtfUy1AXa9X5UYYLKsP_RRw0iLgVCaymOH8LKmm/pub?output=csv'
expert_raw = pd.read_csv(expert_url, header=None)
print(f"Экспертный список загружен. Размер: {expert_raw.shape}")

# Очистка экспертного списка
expert_clean = expert_raw.iloc[7:].copy()
expert_clean.columns = expert_raw.iloc[7].astype(str).str.strip()
expert_clean = expert_clean.reset_index(drop=True)
expert_clean = expert_clean[expert_clean['№'].notna()]
expert_clean = expert_clean[expert_clean['№'].astype(str).str.strip() != '']

def normalize_issn(issn):
    if pd.isna(issn):
        return None
    return str(issn).replace('-', '').replace(' ', '').strip().upper()

def is_medical_journal(asjc_codes):
    if pd.isna(asjc_codes):
        return False
    codes = str(asjc_codes).replace(';', ',').replace(' ', '').split(',')
    for code in codes:
        code = code.strip()
        if code.startswith('27') or code.startswith('29') or code.startswith('30'):
            return True
    return False

# Отбор медицинских журналов из экспертного списка
expert_medical = expert_clean[expert_clean['ASJC Codes'].apply(is_medical_journal)].copy()
print(f"Медицинских журналов по ASJC: {len(expert_medical)}")

expert_medical['ISSN_print_norm'] = expert_medical['ISSN print (Scopus)'].apply(normalize_issn)
expert_medical['E_ISSN_norm'] = expert_medical['E-ISSN (Scopus)'].apply(normalize_issn)

medical_issns = set()
medical_issns.update(expert_medical['ISSN_print_norm'].dropna().tolist())
medical_issns.update(expert_medical['E_ISSN_norm'].dropna().tolist())
print(f"Уникальных ISSN для медицинских журналов: {len(medical_issns)}")

# Фильтрация основного датасета
main_df['Print ISSN_norm'] = main_df['Print ISSN'].astype(str).str.replace('-', '').str.replace(' ', '').str.strip().str.upper()
main_df['Electronic ISSN_norm'] = main_df['Electronic ISSN'].astype(str).str.replace('-', '').str.replace(' ', '').str.strip().str.upper()

main_df = main_df[
    (main_df['Print ISSN_norm'].isin(medical_issns)) |
    (main_df['Electronic ISSN_norm'].isin(medical_issns))
]
print(f"После фильтрации по медицинским ISSN: {len(main_df)} строк")

main_df = main_df[main_df['Year'] >= 2016]
main_df = main_df.drop_duplicates(subset=['Source title', 'Year'])
print(f"После фильтрации по годам (>= 2016) и удаления дубликатов: {len(main_df)} строк")

# Очистка числовых данных
def clean_numeric(value):
    if pd.isna(value):
        return np.nan
    value_str = str(value).strip()
    value_str = value_str.replace('%', '')
    value_str = value_str.replace(',', '.')
    value_str = value_str.replace(' ', '')
    try:
        return float(value_str)
    except ValueError:
        return np.nan

for col in ['P', 'IPP', 'SNIP', '% self cit']:
    if col in main_df.columns:
        main_df[col] = main_df[col].apply(clean_numeric)

main_df = main_df.dropna(subset=['P', 'SNIP', '% self cit'])
print(f"После удаления пропусков в ключевых колонках: {len(main_df)} строк")

# ============================================================
# ПРИНУДИТЕЛЬНОЕ ИСКЛЮЧЕНИЕ НЕМЕДИЦИНСКИХ ЖУРНАЛОВ
# ============================================================

print("\n" + "="*60)
print("ПРОВЕРКА И ИСКЛЮЧЕНИЕ НЕМЕДИЦИНСКИХ ЖУРНАЛОВ")
print("="*60)

# 1. Находим журналы без медицинских ASJC
non_medical_mask = ~main_df['ASJC field IDs'].str.contains('27|29|30', na=False)
non_medical_journals = main_df[non_medical_mask]

if len(non_medical_journals) > 0:
    print(f"\nНайдено {len(non_medical_journals)} записей с немедицинскими ASJC:")
    print(non_medical_journals[['Source title', 'ASJC field IDs', 'Year']].to_string(index=False))

    # 2. Исключаем их из датасета
    main_df = main_df[~non_medical_mask]
    print(f"\n✅ Исключено {len(non_medical_journals)} записей. Новый размер датасета: {len(main_df)}")
else:
    print("\n✅ Все журналы имеют медицинские ASJC. Проверка пройдена.")

# 3. Повторная проверка (должно быть 0)
non_medical_final = main_df[~main_df['ASJC field IDs'].str.contains('27|29|30', na=False)]
print(f"\nФинальная проверка: журналов без медицинских ASJC: {len(non_medical_final)} (должно быть 0)")

# ============================================================
# 2. ТАБЛИЦА 1: ДИНАМИКА ПОКАЗАТЕЛЕЙ (2016–2025)
# ============================================================

print("\n" + "="*60)
print("2. ТАБЛИЦА 1: ДИНАМИКА ПОКАЗАТЕЛЕЙ (2016–2025)")
print("="*60)
print("Примечание: P — среднее количество статей, опубликованных одним журналом за год.")
print("  - 'Среднее P' = среднее арифметическое P по всем журналам в году.")
print("  - 'Суммарное P' = общее количество статей, опубликованных всеми журналами в году.\n")

# Расчет статистик
yearly_stats = main_df.groupby('Year').agg({
    'P': ['mean', 'median', 'count', 'sum'],  # добавляем sum
    'SNIP': ['mean', 'median']
}).round(3)

# Переименовываем колонки
yearly_stats.columns = [
    'Среднее число статей на журнал',
    'Медиана P',
    'Кол-во записей (журналов)',
    'Суммарное P (всего статей)',
    'Средний SNIP',
    'Медиана SNIP'
]

yearly_stats = yearly_stats.reset_index()

# Вывод с разделителями для читаемости
print(yearly_stats.to_string(index=False))
yearly_stats.to_csv('/content/table_dynamics.csv', index=False)

print("\n✅ Таблица сохранена как 'table_dynamics.csv'")

# ============================================================
# 4. ТАБЛИЦА 3: АНАЛИЗ ОТДЕЛЬНЫХ ЖУРНАЛОВ (ВКЛЮЧАЯ НОВЫЕ)
# ============================================================

print("\n" + "="*60)
print("4. ТАБЛИЦА 3: АНАЛИЗ ОТДЕЛЬНЫХ ЖУРНАЛОВ")
print("="*60)
print("Методология: динамика рассчитана для журналов с ≥ 2 годами данных.")
print("  - Журналы с 1 годом включаются в таблицу с пометкой 'new' (без динамики).\n")

def calculate_journal_metrics(group):
    """
    Рассчитывает метрики для журнала. Возвращает словарь с данными.
    Если данных недостаточно (< 2 лет), возвращает базовую информацию без динамики.
    """
    years = group['Year'].values
    n_years = len(group)

    # Базовые данные
    journal_name = group['Source title'].iloc[0]

    # Если только 1 год — возвращаем базовую информацию
    if n_years < 2:
        return {
            'Source title': journal_name,
            'P_start': np.nan,
            'P_end': group['P'].iloc[0],
            'SNIP_start': np.nan,
            'SNIP_end': group['SNIP'].iloc[0],
            'ΔP_abs': np.nan,
            'ΔP_rel': np.nan,
            'ΔSNIP_abs': np.nan,
            'ΔSNIP_rel': np.nan,
            'CV_SNIP': np.nan,
            'n_years': n_years,
            'first_year': years[0],
            'last_year': years[0],
            'status': 'new'
        }

    # Проверка на непрерывность годов (для журналов с ≥ 2 лет)
    if not np.all(np.diff(years) == 1):
        # Если есть пропуски, используем первый и последний год
        p_start = group[group['Year'] == years.min()]['P'].values[0]
        p_end = group[group['Year'] == years.max()]['P'].values[0]
        snip_start = group[group['Year'] == years.min()]['SNIP'].values[0]
        snip_end = group[group['Year'] == years.max()]['SNIP'].values[0]
    else:
        p_start = group[group['Year'] == 2016]['P'].values[0] if 2016 in years else np.nan
        p_end = group[group['Year'] == 2025]['P'].values[0] if 2025 in years else np.nan
        snip_start = group[group['Year'] == 2016]['SNIP'].values[0] if 2016 in years else np.nan
        snip_end = group[group['Year'] == 2025]['SNIP'].values[0] if 2025 in years else np.nan

    return {
        'Source title': journal_name,
        'P_start': p_start,
        'P_end': p_end,
        'SNIP_start': snip_start,
        'SNIP_end': snip_end,
        'ΔP_abs': p_end - p_start,
        'ΔP_rel': ((p_end - p_start) / p_start * 100) if p_start > 0 else np.nan,
        'ΔSNIP_abs': snip_end - snip_start,
        'ΔSNIP_rel': ((snip_end - snip_start) / snip_start * 100) if snip_start > 0 else np.nan,
        'CV_SNIP': (group['SNIP'].std() / group['SNIP'].mean() * 100) if group['SNIP'].mean() > 0 else np.nan,
        'n_years': n_years,
        'first_year': years.min(),
        'last_year': years.max(),
        'status': 'established'
    }

# ============================================================
# 4. ТАБЛИЦА 3: АНАЛИЗ ОТДЕЛЬНЫХ ЖУРНАЛОВ
# ============================================================
# Рассчитываем метрики для всех журналов
journal_metrics = []
for name, group in main_df.groupby('Source title'):
    metrics = calculate_journal_metrics(group)
    if metrics is not None:
        journal_metrics.append(metrics)

journal_df = pd.DataFrame(journal_metrics)
print(f"Всего журналов в таблице: {len(journal_df)}")
print(f"  - Установившиеся (≥ 2 лет): {len(journal_df[journal_df['status'] == 'established'])}")
print(f"  - Новые (1 год): {len(journal_df[journal_df['status'] == 'new'])}")

# Выводим новые журналы
new_journals = journal_df[journal_df['status'] == 'new']
if len(new_journals) > 0:
    print("\nНовые журналы (1 год данных):")
    print(new_journals[['Source title', 'first_year', 'P_end', 'SNIP_end']].to_string(index=False))

# Сохраняем полную таблицу
journal_df.to_csv('/content/table_journal_metrics_full.csv', index=False)
print("\n✅ Полная таблица сохранена как 'table_journal_metrics_full.csv'")

# ============================================================
# ТОП-10 ПО РОСТУ SNIP (только для установившихся журналов)
# ============================================================

established_journals = journal_df[journal_df['status'] == 'established']

top10_snip = established_journals.nlargest(10, 'ΔSNIP_abs')
print("\nТОП-10 ПО РОСТУ SNIP (2016 → 2025, только установившиеся журналы):")
print(top10_snip[['Source title', 'ΔSNIP_abs', 'ΔSNIP_rel']].to_string(index=False))

top10_snip_decline = established_journals.nsmallest(10, 'ΔSNIP_abs')
print("\nТОП-10 ПО ПАДЕНИЮ SNIP (2016 → 2025, только установившиеся журналы):")
print(top10_snip_decline[['Source title', 'ΔSNIP_abs', 'ΔSNIP_rel']].to_string(index=False))



# ============================================================
# ДИАГНОСТИКА: ПОИСК ИСКЛЮЧЁННЫХ ЖУРНАЛОВ
# ============================================================

print("\n" + "="*60)
print("ДИАГНОСТИКА: ПОИСК ИСКЛЮЧЁННЫХ ЖУРНАЛОВ")
print("="*60)

# 1. Получаем список всех уникальных журналов в main_df
all_journals = set(main_df['Source title'].unique())

# 2. Получаем список журналов, для которых рассчитаны метрики
journal_metrics_titles = set(journal_df['Source title'].unique())

# 3. Находим разницу (журналы без метрик)
missing_journals = all_journals - journal_metrics_titles

print(f"\nВсего уникальных журналов в данных: {len(all_journals)}")
print(f"Журналов с рассчитанными метриками: {len(journal_metrics_titles)}")
print(f"Журналов без метрик: {len(missing_journals)}")

if len(missing_journals) > 0:
    print("\nЖурналы без рассчитанных метрик:")
    for journal in sorted(missing_journals):
        # Выводим информацию о годах, в которых встречается журнал
        years = main_df[main_df['Source title'] == journal]['Year'].tolist()
        print(f"  - {journal}: годы {sorted(years)}")

# Топ-10 по росту SNIP
top10_snip = journal_df.nlargest(10, 'ΔSNIP_abs')
print("\nТОП-10 ПО РОСТУ SNIP (2016 → 2025):")
print(top10_snip[['Source title', 'ΔSNIP_abs', 'ΔSNIP_rel']].to_string(index=False))

# Топ-10 по падению SNIP
top10_snip_decline = journal_df.nsmallest(10, 'ΔSNIP_abs')
print("\nТОП-10 ПО ПАДЕНИЮ SNIP (2016 → 2025):")
print(top10_snip_decline[['Source title', 'ΔSNIP_abs', 'ΔSNIP_rel']].to_string(index=False))

journal_df.to_csv('/content/table_journal_metrics.csv', index=False)
print("\n✅ Таблица сохранена как 'table_journal_metrics.csv'")

# ============================================================
# 5. ТАБЛИЦА 4: АНАЛИЗ САМОЦИТИРОВАНИЯ
# ============================================================

print("\n" + "="*60)
print("5. ТАБЛИЦА 4: АНАЛИЗ САМОЦИТИРОВАНИЯ")
print("="*60)

selfcit_journal = main_df.groupby('Source title').agg({
    '% self cit': ['mean', 'max', 'std'],
    'SNIP': 'mean'
}).round(3)
selfcit_journal.columns = ['SelfCit_mean', 'SelfCit_max', 'SelfCit_std', 'SNIP_mean']
selfcit_journal = selfcit_journal.reset_index()

# Топ-10 по самоцитированию
top10_selfcit = selfcit_journal.nlargest(10, 'SelfCit_mean')
print("\nТОП-10 ПО САМОЦИТИРОВАНИЮ:")
print(top10_selfcit[['Source title', 'SelfCit_mean', 'SelfCit_max', 'SNIP_mean']].to_string(index=False))

# Топ-10 с минимальным самоцитированием
bottom10_selfcit = selfcit_journal.nsmallest(10, 'SelfCit_mean')
print("\nТОП-10 С МИНИМАЛЬНЫМ САМОЦИТИРОВАНИЕМ:")
print(bottom10_selfcit[['Source title', 'SelfCit_mean', 'SelfCit_max', 'SNIP_mean']].to_string(index=False))

selfcit_journal.to_csv('/content/table_selfcit.csv', index=False)
print("\n✅ Таблица сохранена как 'table_selfcit.csv'")

# ============================================================
# 6. СТАТИСТИЧЕСКИЕ ПРОВЕРКИ
# ============================================================

print("\n" + "="*60)
print("6. СТАТИСТИЧЕСКИЕ ПРОВЕРКИ")
print("="*60)

# H4.1: Онкология vs Хирургия
oncology = main_df[main_df['ASJC field IDs'].str.contains('2730', na=False)]
surgery = main_df[main_df['ASJC field IDs'].str.contains('2746', na=False)]

if len(oncology) > 5 and len(surgery) > 5:
    t_stat, p_val = ttest_ind(oncology['% self cit'], surgery['% self cit'], equal_var=False)
    print(f"\nH4.1: Онкология vs Хирургия (самоцитирование)")
    print(f"  Онкология: среднее = {oncology['% self cit'].mean():.3f}, n = {len(oncology)}")
    print(f"  Хирургия: среднее = {surgery['% self cit'].mean():.3f}, n = {len(surgery)}")
    print(f"  t-test: t = {t_stat:.3f}, p = {p_val:.4f}")

# Корреляция самоцитирования и CV_SNIP
merged = pd.merge(selfcit_journal, journal_df[['Source title', 'CV_SNIP']], on='Source title')
if len(merged) > 10:
    corr, p = pearsonr(merged['SelfCit_mean'], merged['CV_SNIP'])
    print(f"\nКорреляция самоцитирования и стабильности (CV_SNIP):")
    print(f"  r = {corr:.3f}, p = {p:.4f}")

# ============================================================
# 7. МОДУЛЬ ВЕРИФИКАЦИИ ДАННЫХ
# ============================================================

print("\n" + "="*60)
print("7. МОДУЛЬ ВЕРИФИКАЦИИ ДАННЫХ")
print("="*60)

# 1. Дубликаты
duplicates = main_df.duplicated(subset=['Source title', 'Year']).sum()
print(f"\n1. Дубликаты (журнал + год): {duplicates} (должно быть 0)")

# 2. Пропуски
missing = main_df[['P', 'SNIP', '% self cit']].isna().sum()
print(f"\n2. Пропуски в ключевых колонках:\n{missing.to_string()}")

# 3. Аномалии SNIP
snip_outliers = main_df[(main_df['SNIP'] < 0) | (main_df['SNIP'] > 10)]
print(f"\n3. Аномальные значения SNIP ( < 0 или > 10): {len(snip_outliers)}")

# 4. Медицинские ASJC
non_medical = main_df[~main_df['ASJC field IDs'].str.contains('27|29|30', na=False)]
print(f"\n4. Журналы без медицинских ASJC: {len(non_medical)} (должно быть 0)")

# 5. ISSN
if 'expert_issns' in locals():
    main_issns = set(main_df['Print ISSN_norm'].dropna().tolist() + main_df['Electronic ISSN_norm'].dropna().tolist())
    invalid_issns = main_issns - expert_issns
    print(f"\n5. ISSN, не найденные в экспертном списке: {len(invalid_issns)} (должно быть 0)")

# 6. Диапазон лет
print(f"\n6. Диапазон лет: {main_df['Year'].min()} – {main_df['Year'].max()} (ожидалось 2016–2025)")

# 7. Количество записей по годам
year_counts = main_df.groupby('Year').size()
print(f"\n7. Количество записей по годам:\n{year_counts.to_string()}")
print(f"   Среднее: {year_counts.mean():.1f}, Медиана: {year_counts.median():.0f}")

# 8. Полнота рядов
journal_years = main_df.groupby('Source title')['Year'].nunique()
incomplete = journal_years[journal_years < 10]
print(f"\n8. Журналы с неполным рядом данных (< 10 лет): {len(incomplete)}")

# 9. Самоцитирование > 100%
selfcit_high = main_df[main_df['% self cit'] > 100]
print(f"\n9. Самоцитирование > 100%: {len(selfcit_high)} (должно быть 0)")

# 10. Отрицательное P
p_negative = main_df[main_df['P'] < 0]
print(f"\n10. Отрицательное P: {len(p_negative)} (должно быть 0)")

print("\n✅ ВЕРИФИКАЦИЯ ЗАВЕРШЕНА")

# ============================================================
# 8. АНАЛИЗ ПРИРОСТА ЖУРНАЛОВ ПО ГОДАМ
# ============================================================

print("\n" + "="*60)
print("8. АНАЛИЗ ПРИРОСТА ЖУРНАЛОВ ПО ГОДАМ")
print("="*60)

# Первое появление каждого журнала
first_appearance = main_df.groupby('Source title')['Year'].min().reset_index()
first_appearance.columns = ['Source title', 'first_year']

# Новые журналы по годам
new_journals_by_year = first_appearance.groupby('first_year').size().reset_index(name='new_journals')

# Активные журналы по годам
active_journals_by_year = main_df.groupby('Year')['Source title'].nunique().reset_index(name='active_journals')

# Объединение
growth_df = pd.merge(active_journals_by_year, new_journals_by_year, left_on='Year', right_on='first_year', how='left').fillna(0)
growth_df['new_journals'] = growth_df['new_journals'].astype(int)
growth_df['cumulative'] = growth_df['new_journals'].cumsum()

print("\nРеальная динамика журналов:")
print(growth_df[['Year', 'active_journals', 'new_journals', 'cumulative']].to_string(index=False))

growth_df.to_csv('/content/journals_growth_correct.csv', index=False)
print("\n✅ Таблица сохранена как 'journals_growth_correct.csv'")

# Визуализация прироста
plt.figure(figsize=(14, 6))
sns.barplot(x='Year', y='active_journals', data=growth_df, palette='Blues_d', edgecolor='black')
plt.xlabel('Год', fontsize=12)
plt.ylabel('Количество активных журналов', fontsize=12)
plt.title('Количество российских медицинских журналов, активных в Scopus (2016–2025)', fontsize=14)
plt.grid(True, alpha=0.3)
for i, row in growth_df.iterrows():
    plt.text(row['Year'], row['active_journals'] + 2, str(row['active_journals']), ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/content/active_journals_by_year.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 9. АНАЛИЗ ДИНАМИКИ ПО ASJC ТЕМАТИКАМ (ИСПРАВЛЕННАЯ ВЕРСИЯ)
# ============================================================

print("\n" + "="*60)
print("АНАЛИЗ ДИНАМИКИ ПО ASJC ТЕМАТИКАМ")
print("="*60)

# 1. Разбиваем ASJC коды на отдельные строки
asjc_expanded = []
for _, row in main_df.iterrows():
    asjc_str = row['ASJC field IDs']
    if pd.isna(asjc_str) or str(asjc_str).strip() == '':
        continue
    codes = str(asjc_str).replace(';', ',').replace(' ', '').split(',')
    for code in codes:
        if code.startswith(('27', '29', '30')):
            asjc_expanded.append({
                'ASJC': code,
                'Year': row['Year'],
                'P': row['P'],
                'SNIP': row['SNIP'],
                'Source title': row['Source title']
            })

asjc_detail = pd.DataFrame(asjc_expanded)

# 2. Агрегация по ASJC и годам
asjc_yearly = asjc_detail.groupby(['ASJC', 'Year']).agg({
    'P': 'sum',                  # Суммарное число статей по тематике за год
    'Source title': 'nunique',   # Число журналов, публикующих по этой тематике
    'SNIP': 'mean'               # Средний SNIP по тематике за год
}).reset_index()
asjc_yearly.columns = ['ASJC', 'Year', 'Суммарное P', 'Число журналов', 'Средний SNIP']

# 3. Сравнение периодов 2016–2020 и 2021–2025
period1 = asjc_yearly[asjc_yearly['Year'].between(2016, 2020)]
period2 = asjc_yearly[asjc_yearly['Year'].between(2021, 2025)]

p1_stats = period1.groupby('ASJC').agg({
    'Суммарное P': 'mean',
    'Число журналов': 'mean',
    'Средний SNIP': 'mean'
}).rename(columns={
    'Суммарное P': 'P_2016_2020',
    'Число журналов': 'Journals_2016_2020',
    'Средний SNIP': 'SNIP_2016_2020'
})

p2_stats = period2.groupby('ASJC').agg({
    'Суммарное P': 'mean',
    'Число журналов': 'mean',
    'Средний SNIP': 'mean'
}).rename(columns={
    'Суммарное P': 'P_2021_2025',
    'Число журналов': 'Journals_2021_2025',
    'Средний SNIP': 'SNIP_2021_2025'
})

# 4. Объединяем и рассчитываем изменения
asjc_trend = pd.merge(p1_stats, p2_stats, on='ASJC', how='outer').fillna(0)

# ============================================================
# ВАЖНО: СБРАСЫВАЕМ ИНДЕКС, ЧТОБЫ ASJC СТАЛ КОЛОНКОЙ
# ============================================================
asjc_trend = asjc_trend.reset_index()

asjc_trend['ΔP_abs'] = asjc_trend['P_2021_2025'] - asjc_trend['P_2016_2020']
asjc_trend['ΔP_rel'] = (asjc_trend['ΔP_abs'] / asjc_trend['P_2016_2020'].replace(0, np.nan)) * 100
asjc_trend['ΔJournals_abs'] = asjc_trend['Journals_2021_2025'] - asjc_trend['Journals_2016_2020']
asjc_trend['ΔJournals_rel'] = (asjc_trend['ΔJournals_abs'] / asjc_trend['Journals_2016_2020'].replace(0, np.nan)) * 100
asjc_trend['ΔSNIP_abs'] = asjc_trend['SNIP_2021_2025'] - asjc_trend['SNIP_2016_2020']
asjc_trend['ΔSNIP_rel'] = (asjc_trend['ΔSNIP_abs'] / asjc_trend['SNIP_2016_2020'].replace(0, np.nan)) * 100

asjc_trend = asjc_trend.replace([np.inf, -np.inf], np.nan).round(3)

# 5. Расчёт доли в общем объёме публикаций
total_p_period1 = asjc_trend['P_2016_2020'].sum()
total_p_period2 = asjc_trend['P_2021_2025'].sum()

asjc_trend['Доля_2016_2020'] = (asjc_trend['P_2016_2020'] / total_p_period1 * 100).round(1)
asjc_trend['Доля_2021_2025'] = (asjc_trend['P_2021_2025'] / total_p_period2 * 100).round(1)
asjc_trend['ΔДоля'] = asjc_trend['Доля_2021_2025'] - asjc_trend['Доля_2016_2020']

# 6. Сортируем по абсолютному приросту статей
asjc_trend_sorted = asjc_trend.sort_values('ΔP_abs', ascending=False)

# 7. Вывод ТОП-10 и BOTTOM-10
print("\nТОП-10 ТЕМАТИК ПО РОСТУ ЧИСЛА СТАТЕЙ (абсолютный прирост):")
top10_p = asjc_trend_sorted.head(10)
print(top10_p[['ASJC', 'P_2016_2020', 'P_2021_2025', 'ΔP_abs', 'ΔP_rel', 'Доля_2016_2020', 'Доля_2021_2025', 'ΔДоля']].to_string(index=False))

print("\nТОП-10 ТЕМАТИК ПО ПАДЕНИЮ ЧИСЛА СТАТЕЙ (абсолютный спад):")
bottom10_p = asjc_trend_sorted.tail(10)
print(bottom10_p[['ASJC', 'P_2016_2020', 'P_2021_2025', 'ΔP_abs', 'ΔP_rel', 'Доля_2016_2020', 'Доля_2021_2025', 'ΔДоля']].to_string(index=False))

# 8. Вывод ТОП-10 по росту числа журналов
print("\nТОП-10 ТЕМАТИК ПО РОСТУ ЧИСЛА ЖУРНАЛОВ:")
top10_journals = asjc_trend_sorted.nlargest(10, 'ΔJournals_abs')
print(top10_journals[['ASJC', 'Journals_2016_2020', 'Journals_2021_2025', 'ΔJournals_abs', 'ΔJournals_rel']].to_string(index=False))

# 9. Вывод ТОП-10 по росту SNIP
print("\nТОП-10 ТЕМАТИК ПО РОСТУ SNIP:")
top10_snip = asjc_trend_sorted.nlargest(10, 'ΔSNIP_abs')
print(top10_snip[['ASJC', 'SNIP_2016_2020', 'SNIP_2021_2025', 'ΔSNIP_abs', 'ΔSNIP_rel']].to_string(index=False))

# 10. Сохраняем таблицу
asjc_trend_sorted.to_csv('/content/table_asjc_detailed_dynamics.csv', index=False)
print("\n✅ Полная таблица сохранена как 'table_asjc_detailed_dynamics.csv'")
print("\n" + "="*60)
print("АНАЛИЗ ДИНАМИКИ ПО ASJC ТЕМАТИКАМ")
print("="*60)

# ============================================================
# КОРРЕЛЯЦИЯ МЕЖДУ РОСТОМ ЧИСЛА СТАТЕЙ И РОСТОМ SNIP ПО ТЕМАТИКАМ
# ============================================================

print("\n" + "="*60)
print("КОРРЕЛЯЦИЯ: РОСТ СТАТЕЙ vs РОСТ SNIP ПО ТЕМАТИКАМ")
print("="*60)

# Используем таблицу asjc_trend_sorted (уже рассчитана)
# Удаляем строки, где нет данных по SNIP (было 0 в начальном периоде)
corr_data = asjc_trend_sorted[
    (asjc_trend_sorted['ΔP_abs'].notna()) &
    (asjc_trend_sorted['ΔSNIP_abs'].notna()) &
    (asjc_trend_sorted['P_2016_2020'] > 0)  # исключаем тематики с нулевым стартом
]

# Расчет корреляции Пирсона
if len(corr_data) > 2:
    corr_coef, p_value = pearsonr(corr_data['ΔP_abs'], corr_data['ΔSNIP_abs'])
    print(f"\nКоэффициент корреляции Пирсона (ΔP_abs vs ΔSNIP_abs): r = {corr_coef:.3f}, p = {p_value:.4f}")

    if p_value < 0.05:
        print("  → Корреляция статистически значима (p < 0.05).")
    else:
        print("  → Корреляция статистически не значима (p ≥ 0.05).")

    if corr_coef > 0.5:
        print("  → Сильная положительная связь: рост числа статей сопровождается ростом SNIP.")
    elif corr_coef > 0.3:
        print("  → Умеренная положительная связь: рост числа статей частично сопровождается ростом SNIP.")
    elif corr_coef > 0.1:
        print("  → Слабая положительная связь: рост числа статей слабо связан с ростом SNIP.")
    elif corr_coef > -0.1:
        print("  → Связь отсутствует (практически нулевая).")
    elif corr_coef > -0.3:
        print("  → Слабая отрицательная связь: рост числа статей слабо связан со снижением SNIP.")
    else:
        print("  → Умеренная или сильная отрицательная связь: рост числа статей сопровождается снижением SNIP.")
else:
    print("\nНедостаточно данных для расчета корреляции.")

# ============================================================
# ДОПОЛНИТЕЛЬНЫЙ АНАЛИЗ: КОРРЕЛЯЦИЯ ДЛЯ ТОП-20 ТЕМАТИК
# ============================================================

print("\n" + "-"*60)
print("КОРРЕЛЯЦИЯ ДЛЯ ТОП-20 ТЕМАТИК ПО РОСТУ СТАТЕЙ")
print("-"*60)

top20_data = asjc_trend_sorted.head(20)
top20_data = top20_data[
    (top20_data['ΔP_abs'].notna()) &
    (top20_data['ΔSNIP_abs'].notna()) &
    (top20_data['P_2016_2020'] > 0)
]

if len(top20_data) > 2:
    corr_coef_top20, p_value_top20 = pearsonr(top20_data['ΔP_abs'], top20_data['ΔSNIP_abs'])
    print(f"Коэффициент корреляции для ТОП-20: r = {corr_coef_top20:.3f}, p = {p_value_top20:.4f}")

    if p_value_top20 < 0.05:
        print("  → Корреляция статистически значима.")
    else:
        print("  → Корреляция статистически не значима.")

# ============================================================
# ТОП-5 ТЕМАТИК С НАИБОЛЬШИМ СООТНОШЕНИЕМ ΔSNIP / ΔP
# ============================================================

print("\n" + "-"*60)
print("ТОП-5 ТЕМАТИК С НАИБОЛЬШИМ ПРИРОСТОМ SNIP НА ЕДИНИЦУ РОСТА СТАТЕЙ")
print("-"*60)

# Исключаем деление на ноль
ratio_data = asjc_trend_sorted[
    (asjc_trend_sorted['ΔP_abs'] > 10) &  # исключаем тематики с очень малым ростом
    (asjc_trend_sorted['ΔSNIP_abs'].notna()) &
    (asjc_trend_sorted['P_2016_2020'] > 0)
].copy()

ratio_data['SNIP_per_P'] = ratio_data['ΔSNIP_abs'] / ratio_data['ΔP_abs']
ratio_data = ratio_data.sort_values('SNIP_per_P', ascending=False)

top5_ratio = ratio_data.head(5)
print(top5_ratio[['ASJC', 'ΔP_abs', 'ΔSNIP_abs', 'SNIP_per_P']].to_string(index=False))

# ============================================================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================================

# Сохраняем данные для визуализации
corr_data.to_csv('/content/correlation_data_asjc.csv', index=False)
print("\n✅ Данные для корреляции сохранены как 'correlation_data_asjc.csv'")

# ============================================================
# 10. ИТОГОВОЕ СООБЩЕНИЕ
# ============================================================

print("\n" + "="*60)
print("✅ ВСЕ РАСЧЕТЫ ЗАВЕРШЕНЫ")
print("="*60)

print("\nСгенерированные файлы:")
print(" - table_dynamics.csv (динамика показателей)")
print(" - table_asjc_dynamics.csv (динамика тематик)")
print(" - table_journal_metrics.csv (метрики журналов)")
print(" - table_selfcit.csv (самоцитирование)")
print(" - journals_growth_correct.csv (прирост журналов)")
print(" - active_journals_by_year.png (график активных журналов)")

print("\n" + "="*60)
print("МЕТОДОЛОГИЧЕСКИЕ ПРИМЕЧАНИЯ:")
print("="*60)
print("1. Отбор журналов — только по медицинским ASJC (27xx, 29xx, 30xx).")
print("2. Для тематик динамика — разница СРЕДНИХ за периоды 2016–2020 и 2021–2025.")
print("3. Для журналов динамика — разница значений в 2016 и 2025 гг.")
print("4. NaN в Δ_rel означает, что начальное значение = 0 (деление на ноль).")
print("5. Для ранжирования используется абсолютный прирост (Δ_abs).")
print("6. Верификация данных пройдена — все ключевые проверки выполнены.")

1. ЗАГРУЗКА И ФИЛЬТРАЦИЯ ДАННЫХ
Загрузка основного датасета...
Основной датасет загружен. Размер: (606207, 15)
Загрузка экспертного списка...
Экспертный список загружен. Размер: (873, 18)
Медицинских журналов по ASJC: 225
Уникальных ISSN для медицинских журналов: 416
После фильтрации по медицинским ISSN: 2021 строк
После фильтрации по годам (>= 2016) и удаления дубликатов: 1603 строк
После удаления пропусков в ключевых колонках: 1603 строк

ПРОВЕРКА И ИСКЛЮЧЕНИЕ НЕМЕДИЦИНСКИХ ЖУРНАЛОВ

Найдено 3 записей с немедицинскими ASJC:
                              Source title ASJC field IDs  Year
Voprosy Istorii Estestvoznaniia i Tekhniki           1207  2023
Voprosy Istorii Estestvoznaniia i Tekhniki           1207  2024
Voprosy Istorii Estestvoznaniia i Tekhniki           1207  2025

✅ Исключено 3 записей. Новый размер датасета: 1600

Финальная проверка: журналов без медицинских ASJC: 0 (должно быть 0)

2. ТАБЛИЦА 1: ДИНАМИКА ПОКАЗАТЕЛЕЙ (2016–2025)
Примечание: P — среднее количество статей,


АНАЛИЗ ДИНАМИКИ ПО ASJC ТЕМАТИКАМ

ТОП-10 ТЕМАТИК ПО РОСТУ ЧИСЛА СТАТЕЙ (абсолютный прирост):
ASJC  P_2016_2020  P_2021_2025  ΔP_abs  ΔP_rel  Доля_2016_2020  Доля_2021_2025  ΔДоля
2746       1949.0       6373.2  4424.2 226.998             6.3             7.6    1.3
2700       1238.8       4471.0  3232.2 260.914             4.0             5.4    1.4
2701        806.8       3532.4  2725.6 337.828             2.6             4.2    1.6
2739       1226.6       3740.2  2513.6 204.924             4.0             4.5    0.5
2730       1219.0       3454.8  2235.8 183.413             3.9             4.1    0.2
2705       2055.0       4195.4  2140.4 104.156             6.6             5.0   -1.6
2723        715.8       2799.0  2083.2 291.031             2.3             3.4    1.1
2729       1345.8       3394.8  2049.0 152.251             4.4             4.1   -0.3
2725       1216.2       3242.2  2026.0 166.584             3.9             3.9    0.0
2735       1375.2       3397.4  2022.2 147.04